# ***World-model training with data-quality review***

A world model learns to predict what comes next from recent measurements and recorded treatments. For example, it may use the past hour of pH, oxygen, light and dosing to predict the next reading. Sampling interval, history duration, prediction duration and controller update interval are separate choices. An hour of five-second observations contains 720 readings; an hour of minute observations contains 60.

The prediction set contains TDS, DO, pH, COD, TOC, turbidity, water ammonia, water temperature, measured light, biomass estimate, BOD and TSS. **pH, COD, BOD, DO and water ammonia are the priority group.** They receive a separate report without an invented ranking or additional loss weights. The complete target set still determines checkpoint selection.

Interpolated biomass is not a continuous independent measurement. The README documents COD-derived BOD and a dedicated newer-hardware TSS sensor; BOD remains independently predicted at the user's request, and measured TSS history is a required input. For example, accurately reproducing an interpolated biomass curve does not establish that a controller increases actual biomass. Biomass and BOD remain output-only. TSS history is included; its output remains a level prediction, preserving the original target objective. Logged future treatments describe conditional historical prediction, rather than evidence about arbitrary new actions.

Source files remain intact. Quality screens identify readings worth inspecting without deleting them. The default recipe keeps original readings; a separate causal-filter experiment can be enabled. A causal filter only uses measurements already available: a short trailing median may suppress an isolated jump, but may also suppress a real fast response.

Cultivations are separated before creating histories. A held-out validation cultivation selects checkpoints, while the final test stays reserved. Smaller exploratory experiments use starts spread across the episode; the uncapped alternative uses every eligible start at native start spacing. All families share the same declared training cohort within a comparison. Memory guards can skip fits that do not fit the budget, and histories are constructed on demand.

Completed training saves a model, frozen transformations, source hashes, forecast starts and data-quality settings. Evaluation reloads these artifacts for comparisons, uncertainty and longer recursive forecasts. Training does not establish controller performance, treatment compliance or independent biological validity.

The capped option reserves a declared allocation for logged treatment events; the current full_scalable configuration uses all eligible starts at native spacing. All learned families receive the same selected starts. The larger track remains available. Observations inside every history retain their original cadence; increasing the spacing of training starts is a separate sampling experiment, not downsampling the sensor stream.

## ***Import***

Rerunning this section refreshes the local experiment helpers from this project, including their shared protocol. This prevents an existing kernel from retaining older helper code after notebook updates. Run the notebook from the top after an update.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO
import importlib
import json
import sys
import time
import traceback

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT_OVERRIDE = None
def locate_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'dev' / 'AGENTS.md').is_file():
            return candidate
    raise FileNotFoundError('Set PROJECT_ROOT_OVERRIDE to the project directory.')

PROJECT_ROOT = locate_root()
WORLD_MODEL_DIR = PROJECT_ROOT / 'dev' / 'world_model'
sys.path.insert(0, str(WORLD_MODEL_DIR))
DATASET_EDA_DIR = PROJECT_ROOT / 'dev' / 'dataset_eda'
sys.path.insert(0, str(DATASET_EDA_DIR))
importlib.invalidate_caches()

def refresh_local_helper(name):
    expected = (WORLD_MODEL_DIR / f'{name}.py').resolve()
    cached = sys.modules.get(name)
    if cached is not None and Path(getattr(cached, '__file__', '') or '').resolve() != expected:
        del sys.modules[name]
        cached = None
    module = importlib.reload(cached) if cached is not None else importlib.import_module(name)
    if Path(module.__file__).resolve() != expected:
        raise ImportError(f'{name} was loaded from {module.__file__}; expected {expected}.')
    return module

wp = refresh_local_helper('world_model_protocol')
refresh_local_helper('world_model_recovery')
wm = refresh_local_helper('world_models')
wd = refresh_local_helper('world_model_training_diagnostics')
print('World-model modules loaded from:', WORLD_MODEL_DIR)
display(wm.availability())
try:
    import matplotlib.pyplot as plt
    PLOTTING_AVAILABLE = True
except ImportError as error:
    PLOTTING_AVAILABLE = False
    print('Plotting unavailable:', error)

if wm.TQDM_AVAILABLE:
    from tqdm.auto import tqdm
    from tqdm import tqdm as text_tqdm
print('tqdm progress available:', wm.TQDM_AVAILABLE)
if not wm.TQDM_AVAILABLE:
    print('Install tqdm in this kernel environment to show progress; training can still run without bars.')


World-model modules loaded from: /home/rahmadbisma/Documents/ITS/MBRL_MicroAlgae/dev/world_model


,package,available,version,import_error
0,torch,True,2.12.0+cu130,NaN
1,sklearn,True,1.8.0,NaN
2,xgboost,True,3.2.0,NaN
3,lightgbm,False,NaN,No module named 'lightgbm'
4,scipy,True,1.17.1,NaN
5,joblib,True,1.5.3,NaN


tqdm progress available: True


## ***Config***

The device report describes the Python kernel running this notebook. For example, automatic selection uses an accessible CUDA GPU for neural training and otherwise reports why it falls back to CPU. A CUDA-enabled PyTorch installation still needs access to the GPU driver and device. A restricted checking environment can therefore report no GPU even when a normal local kernel can use it.

Each model section displays its settings and intended execution device before fitting, then reports the device returned by a successful fit. The current scikit-learn estimators, XGBoost histogram implementation and physical calibration run on CPU, even when neural training uses a GPU. A displayed configuration does not mean that a model passed its dependency or memory checks.

Terminal text progress is the default so batch counts also reach Papermill logs, tmux and redirected output. For example, the batch counter can advance in a log while a notebook widget would remain at its initial text representation. Automatic widget selection remains available for interactive notebook use. This display choice does not change sampling, optimization or checkpoints.


**Training rollouts:** `TRAINING_ROLLOUT_STEPS` is a count of sampling steps, not a duration, so its meaning depends on `SAMPLING_INTERVAL_SECONDS`. At one-minute cadence `15` is the 15-minute planning horizon; at five-second cadence the same 15 steps is 75 seconds, and 15 minutes would need `180`. Use `1` for the original one-step objective. This is separate from input-history length, which is expressed in `history_minutes`, and from the one-step inference horizon. `ROLLOUT_CURRICULUM` optionally stages the length by epoch; its final stage must reach the configured maximum. The complete maximum-length cohort is used throughout the curriculum.

Multi-step training repeats the network forward pass at every rollout step and can take much longer than one-step training. Activation checkpointing reduces saved activations by recomputing them during backward; lower `batch_size` if memory is insufficient. It does not detach prediction feedback. New runs receive unique IDs; existing checkpoints are not overwritten.


**TCN head experiment:** `CONFIG['neural']['tcn']['head_layout']` selects the output layout. `shared` uses one head for all 12 targets; `dedicated` uses one shared head for the other nine targets plus individual BOD, biomass-estimate and TSS heads. `latest_sensor_to_head=True` also concatenates the latest normalized history values for measured target sensors directly into the prediction head. Set either option independently to compare the variants; the current TCN configuration uses dedicated heads while the latest-sensor connection is off, keeping experiments 7 and 8 separable. These options change the architecture and require a fresh run; the saved architecture metadata lets evaluation reconstruct it. Batch size, hidden size, learning rate and rollout settings remain independently configured above.


**Independent randomness:** change `seed` between training repeats; keep `sampling_seed=42` fixed. The latter selects training windows, checkpoint-validation windows and the separate treatment audit. The neural trainer still receives the original `CONFIG`, so initialization, shuffling, dropout and ensemble bootstrap use the training/member seeds. Saved config and anchors record both roles. Fixed sampling does not force different ensemble members to use identical bootstrap draws.


### ***Interruption recovery and response diagnostics***

Full-state recovery is enabled by default and writes a separate `.recovery.pt` file after each completed epoch and ensemble member. It saves the current and best model, optimizer, scheduler, early-stopping state, data-loader/RNG states, completed members, curve and experiment identity. No mixed precision is currently used; its state slot is explicitly empty. An unfinished epoch is replayed after interruption; recovery is not mid-batch continuation. Set `CONFIG['recovery']['resume_by_algorithm']['tcn']` to the printed exact local file path (`None` starts fresh) and rerun from Import with unchanged training settings. Completed inference artifacts remain separate, and weight-only warm start still resets optimizer/scheduler. Checkpoint files use atomic replacement; a mismatch in data, preprocessing, validation windows, settings, code or runtime blocks recovery. Load only trusted local recovery files.

Selected-checkpoint response diagnostics use a small fixed K14 cohort of logged action events and no-event segments. They recursively feed predicted sensors, persist unpredicted environment, and replay actions/clock. Response amplitude is maximum absolute change from the last observed target value; delay is the first crossing of a configurable fraction of the observed response amplitude, not a causal biological lag estimate. Simultaneous actions and later interventions can confound these descriptions. These diagnostics do not alter checkpoint selection, scheduler or loss. Per-channel validity includes the rollout step and negative magnitude. Full independent model comparisons remain in evaluation.ipynb.

The dataset README defines newer TSS as a measured water-quality channel. All new runs use 21 history channels including measured TSS. TSS is inserted before the five actions, remains a level target, and its predictions feed recursive history. Neither targets nor effective per-target loss weights are changed. BOD remains a separate predicted target at the user's request; no COD/2 constraint is introduced. Sensor timestamps are post-treatment readings; action exposure within a five-second interval can vary.


This experimental backend saves and loads `world_model_v5` only. Existing older artifacts remain untouched; no migration or legacy loading is provided. The former input schema without TSS is no longer supported.


In [ ]:
# Papermill parameters: overridden per leave-one-cultivation-out fold by run_loco.py.
# Editing these here changes only the interactive default run.
SAMPLING_INTERVAL_SECONDS = 60
SEED = 44
PREFIX = 'wm_12targets_v6'
# Every algorithm the registry knows, grouped as in the notebook sections.
# Uncomment what you want; an unavailable dependency is skipped with a recorded reason.
NEURAL_ALGORITHMS = ['gru', 'lstm', 'tcn', 'pg_ac_tcn', 'pets_mlp', 'dlinear', 'nhits', 'tft',
                      'dnlssm', 'mopo_ensemble', 'moup_mlp']
CLASSICAL_ALGORITHMS = ['ridge', 'random_forest', 'hrp_xgb', 'lightgbm', 'xgboost', 'gpr']
# PHYSICAL_ALGORITHMS = ['grey_box']
ALL_ALGORITHMS = NEURAL_ALGORITHMS + CLASSICAL_ALGORITHMS

ALGORITHMS = ['tcn']  # edit this, or use one of the lists above
CULTIVATIONS = [11, 12, 13, 14, 16]  # K15 excluded: 45 h, README marks it incomplete
FOLD_VALIDATION = 13  # None = hold out the last cultivation; an int = that LOCO fold;
                        # 'all' = no held-out set (final all-data model; needs fixed_epochs)
RUN_ID = None  # None derives the standard run id; run_loco.py passes an explicit fold id
SELECTION = 'validation'  # 'validation' early-stops on the fold; 'fixed_epochs' for the all-data model
FIXED_EPOCHS = None  # required when SELECTION='fixed_epochs'; derive it from the LOCO fold epochs


In [ ]:
pass  # SAMPLING_INTERVAL_SECONDS is set in the parameters cell above
LABEL_MODE = 'proxy_prototype'  # supplied CSV labels; not verified biological outcomes
TRAINING_TRACK = 'full_scalable'  # shared pilot; choose full_scalable for the larger experiment
pass  # SEED is set in the parameters cell above
SAMPLING_SEED = 42  # fixed training/checkpoint/audit window selection across runs
TARGET_PROVENANCE = {
    '1_tds(ppm)': 'TDS sensor proxy; calibration unresolved',
    '1_toc(mg/L)': 'TOC sensor proxy; independence and calibration unresolved',
    '1_temp_air(C)': 'Water temperature sensor channel; calibration unresolved',
    '2_light(Lux)': 'Measured light sensor channel; distinct from lamp PWM',
    'biomassa_estimasi(g/L)': 'Interpolated OD-derived biomass estimate; future-informed label; output only',
    '1_bod(mg/L)': 'README: firmware BOD derived from COD; retained as independent learned target; output only',
    '1_tss(mg/L)': 'README: dedicated newer-hardware TSS sensor; measured history included; level target retained',
    '1_cod(mg/L)': 'Sensor COD proxy; independent laboratory validation unresolved',
    '1_nh3(mg/L)': 'Water-ammonia sensor proxy; species/reporting basis unresolved',
    '1_turbidity(NTU)': 'Turbidity sensor proxy; not specific to algal biomass',
    '1_ph': 'pH sensor channel; calibration unresolved',
    '1_do(mg/L)': 'Dissolved-oxygen sensor channel; calibration unresolved',
}
TRAINING_ROLLOUT_STEPS = 1  # 15 = the 15-minute planning horizon at one-minute cadence; 1 restores one-step training
ROLLOUT_CURRICULUM = []  # optional: [{'start_epoch': 1, 'steps': 1}, {'start_epoch': 3, 'steps': 12}]

NEURAL_COMMON = {
    # 'validation' early-stops on the held-out cultivation (LOCO folds).
    # 'fixed_epochs' trains for FIXED_EPOCHS and keeps the last epoch (final all-data model);
    # that budget must come from cross-validation evidence, never from training error.
    'selection': SELECTION, 'fixed_epochs': FIXED_EPOCHS,
    'epochs':10, 'patience': 10, 'min_delta': 1e-6,
    'batch_size': 256,
    'rollout_steps': TRAINING_ROLLOUT_STEPS,
    'rollout_curriculum': ROLLOUT_CURRICULUM,
    'rollout_discount': 1.,
    'rollout_checkpointing': False,
    'learning_rate': 1e-4, 'optimizer': 'adamw', 'weight_decay': 1e-4,
    'head_dropout': .1, 'gradient_clip_norm': 1.,
    'loss': 'normalized_mse_plus_nll', 'nll_weight': .01,
    'initialization': 'reference', 'include_initial_checkpoint': True,
    'lr_scheduler': True, 'scheduler_factor': .5, 'scheduler_patience': 3,
    'min_learning_rate': 1e-6,
}

# A LOCO fold holds one cultivation out for selection and fits on the rest.
def fold_split(cultivations, validation_cultivation):
    cultivations = list(cultivations)
    if len(set(cultivations)) != len(cultivations):
        raise ValueError('Cultivation list must be unique.')
    if len(cultivations) < 2:
        raise ValueError('A fold needs at least two cultivations.')
    if validation_cultivation == 'all':
        # Final all-data model: nothing held out, so training must use a fixed epoch
        # budget (selection='fixed_epochs'). There is no held-out score to report.
        return {'train': cultivations, 'validation': [], 'test_reserved': []}
    if validation_cultivation is None:
        # Interactive default: hold out the last cultivation; run_loco.py always sets a fold.
        validation_cultivation = cultivations[-1]
    if validation_cultivation not in cultivations:
        raise ValueError(f'Fold validation {validation_cultivation} is not in the cultivation list.')
    train = [k for k in cultivations if k != validation_cultivation]
    if not train:
        raise ValueError('A fold needs at least one training cultivation.')
    return {'train': train, 'validation': [validation_cultivation], 'test_reserved': []}

CONFIG = {
    'neural': {
            'gru': {**NEURAL_COMMON, 'hidden_size': 48},
            'lstm': {**NEURAL_COMMON, 'hidden_size': 48},
            'tcn': {**NEURAL_COMMON, 'hidden_size': 24, 'members': 1,
                    'head_layout': 'dedicated', 'latest_sensor_to_head': True},
            'pg_ac_tcn': {**NEURAL_COMMON, 'hidden_size': 24, 'members': 1,
                          'head_layout': 'dedicated',
                          'physics_loss_weight': 0., 'physics_bounds': {}, 'initialization': 'reference'},
            # Offline-RL dynamics models (MOPO / MOUP). Each is a probabilistic
            # ensemble; they differ only in how member disagreement becomes uncertainty.
            # The reward penalty, rollout policy and MMD constraint in those papers act on
            # a reward function and an actor-critic, which belong in dev/rl_controller.
            'mopo_ensemble': {**NEURAL_COMMON, 'hidden_size': 64, 'members': 5, 'mc_passes': None},
            'moup_mlp': {**NEURAL_COMMON, 'hidden_size': 64, 'members': 5, 'mc_passes': 10,
                         'mc_dropout': .1},
            'pets_mlp': {**NEURAL_COMMON, 'hidden_size': 32, 'members': 3,
                         'learning_rate': 1e-4, 'head_dropout': .2},
            'dlinear': {**NEURAL_COMMON, 'hidden_size': 64, 'members': 1, 'moving_average_kernel': 25,
                        'initialization': 'reference'},
            'nhits': {**NEURAL_COMMON, 'hidden_size': 128, 'members': 1, 'pool_scales': [1, 4, 16],
                      'initialization': 'reference'},
            'tft': {**NEURAL_COMMON, 'hidden_size': 64, 'members': 1, 'batch_size': 32, 'attention_heads': 4,
                    'initialization': 'reference'},
            'dnlssm': {**NEURAL_COMMON, 'hidden_size': 64, 'members': 1, 'batch_size': 32, 'latent_size': 16,
                       'initialization': 'random'},
    },
    'classical': {
        'ridge': {'alpha_candidates': [100., 1000., 10000., 100000., 1000000., 10000000.],
                    'solver': 'cholesky', 'fit_intercept': True},
        'random_forest': {'n_estimators': 160, 'max_depth': 12, 'min_samples_leaf': 10,
                            'max_features': 'sqrt', 'max_samples': .75, 'bootstrap': True},
        'hrp_xgb': {'members': 5, 'xgb_params': {'n_estimators': 500, 'max_depth': 3, 'learning_rate': .03,
                      'subsample': .8, 'colsample_bytree': .6, 'min_child_weight': 10,
                      'reg_lambda': 20., 'reg_alpha': .1, 'max_bin': 128}},
        'lightgbm': {'n_estimators': 500, 'max_depth': 6, 'num_leaves': 15, 'learning_rate': .03,
                    'min_child_samples': 30, 'subsample': .8, 'subsample_freq': 1, 'colsample_bytree': .8,
                    'reg_lambda': 10., 'verbosity': -1},
        'xgboost': {'n_estimators': 500, 'max_depth': 3, 'learning_rate': .03,
                    'subsample': .8, 'colsample_bytree': .6, 'min_child_weight': 10,
                    'reg_lambda': 20., 'reg_alpha': .1, 'max_bin': 128,
                    'early_stopping_rounds': 25, 'eval_metric': 'rmse'},
        'gpr': {'persistence_prior': True,
                'kernel_candidates': [{'length_scale_factor': length, 'noise_level': noise}
                                        for length in [.5, 1., 2.] for noise in [.05, .2]]},
    },
    'physical': {'multistarts': 3, 'max_nfev': 150},
    
    'dataset_path': 'dev/dataset',
    'model_path': 'dev/world_model/model',
    'prefix': PREFIX,
    'run_id': (RUN_ID if RUN_ID else
               f'{SAMPLING_INTERVAL_SECONDS}S_BT{NEURAL_COMMON["batch_size"]}_SD{SEED}_ROUT{TRAINING_ROLLOUT_STEPS}_'
               + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')),
    'algorithms': list(ALGORITHMS),
    'split': (fold_split(CULTIVATIONS, FOLD_VALIDATION)),
    'history_minutes': 60, #300
    # Delta predictions anchor on a trailing mean instead of the single last reading.
    # At 15 min a 15-step mean predicts the level with ~+0.19 skill over the last
    # value on 8 of 9 measured channels, so the model stops modelling anchor noise.
    'baseline_window_minutes': 15,
    'sampling_interval_seconds': SAMPLING_INTERVAL_SECONDS,
    'training_track': TRAINING_TRACK,
    'window_sampling': 'temporal_strata',
    'training_sampling': {'event_fraction': .25, 'start_spacing_seconds': SAMPLING_INTERVAL_SECONDS,
                          'spacing_candidates_seconds': sorted(set([SAMPLING_INTERVAL_SECONDS,
                              max(30, SAMPLING_INTERVAL_SECONDS), max(60, SAMPLING_INTERVAL_SECONDS)]))},
    'recovery': {'enabled': True, 'every_epochs': 5,
                 'directory': 'dev/world_model/model/recovery',
                 'resume_by_algorithm': {'tcn': None}},  # None starts fresh; set a .recovery.pt path to recover.
    # RL usability gate. Level accuracy is not enough: a policy explores actions, so
    # the model must show a response distinguishable from the no-event control.
    # Direction agreement near 0.5 means the action effect was not learned.
    # Action novelty asks "has this control input been tried?" -- the question a policy
    # actually needs. It needs one model and cannot collapse to a constant on a slow
    # process the way ensemble disagreement does (measured ratio 1.03 state vs 3.80 action).
    'action_novelty': True, 'action_novelty_k': 5,
    'action_response_gate': {'enabled': True, 'min_direction_agreement': .60,
                             'min_events_per_action': 10, 'horizon_minutes': 15},
    'response_diagnostics': {'enabled': True, 'horizons_minutes': [1, 5, 15],
                             'events_per_action': 2, 'no_event_starts': 4,
                             'response_fraction': .1, 'direction_tolerance': 1e-8},
    'action_semantics': 'Post-treatment reading: action at 10:00:00/01 precedes reading at 10:00:05; transition uses logged action at predicted reading; within-bin exposure duration uncertain; logged-action conditional replay',
    'dataset_definition_readme': 'dataset/Data Kultivasi Mikroalga K1-K16/README.md',
    'treatment_diagnostics': {'windows_per_cultivation': 256, 'post_event_minutes': 15},
    'output_dependency_pairs': [('1_turbidity(NTU)', '1_tss(mg/L)'),
                                ('1_cod(mg/L)', '1_toc(mg/L)'), ('1_cod(mg/L)', '1_bod(mg/L)')],
    'priority_targets': wp.PRIORITIES.copy(),
    # Measured sensor channels only. These drive checkpoint selection, so algebraic or
    # proxy columns cannot mask the channels a controller actually regulates.
    # 2_light is excluded: it is lamp PWM scaled by the diurnal cycle, so it is
    # near-deterministic arithmetic rather than dynamics, and it swamped the macro
    # (measured -36x persistence on one channel while the other eight were at parity).
    'headline_targets': ['1_tds(ppm)', '1_do(mg/L)', '1_ph', '1_cod(mg/L)', '1_toc(mg/L)',
                         '1_turbidity(NTU)', '1_nh3(mg/L)', '1_temp_air(C)'],
    # Verified on all 16 cultivations: tss = 1.4999999 * turbidity (residual ~2e-5 of level std).
    # The value is derived at inference and excluded from the loss; the output slot remains
    # so the recursive-feedback contract is unchanged. BOD and biomass stay learned: their
    # relations carry 4-15% real residual and AGENTS.md keeps BOD independent.
    'derived_outputs': {'1_tss(mg/L)': {'source': '1_turbidity(NTU)', 'scale': 1.5, 'offset': 0.0}},
    'point_loss_weight_bounds': [0.5, 2.0],
    'input_filter': {'mode': 'none', 'window_seconds': 3 * SAMPLING_INTERVAL_SECONDS,
                     'features': ['1_cod(mg/L)', '1_toc(mg/L)', '1_turbidity(NTU)']},
    'quality_diagnostics': {'enabled': True, 'median_seconds': 180, 'mad_multiplier': 6.,
                            'return_fraction': .25, 'flat_minutes': 10., 'ph_screen': [0., 14.]},
    'training_recipe': 'treatment_aware_12targets_v4_derived_tss',
    'label_mode': LABEL_MODE,
    'target_transform': 'mixed',
    'target_provenance': TARGET_PROVENANCE,
    'recursive_context_policy': 'persist unpredicted sensor/context channels; replay logged actions',
    'capped_windows_per_cultivation': 400,
    'ram_budget_gib': 8,
    'vram_budget_gib': 6,
    'loader_workers': 0,
    'pin_memory': True,
    'validation_batch_size': 32 if SAMPLING_INTERVAL_SECONDS == 5 else 128,
    'progress_bars': False,  # Print epoch starts and estimated epoch duration instead of bars.
    'progress_backend': 'text',  # text for Papermill/tmux logs; auto for notebook widgets
    'classical_cache_batch_size': 256,
    'classical_working_copy_factor': 4,
    'classical_working_budget_gib': 3,
    'cache_path': 'dev/world_model/model/cache',
    'cleanup_feature_cache': True,
    'rf_tree_chunk_size': 20,
    'gpr_max_train_windows': 1500,
    'gpr_working_budget_gib': 1,
    'validation_windows_per_cultivation': 512,
    'evaluation_working_budget_gib': 1,
    'sampling_block_minutes': 10,
    'seed': SEED,  # model initialization, shuffle, dropout and member bootstrap
    'sampling_seed': SAMPLING_SEED,  # fixed training/checkpoint/audit window selection across runs
    'device': 'auto',
    'cpu_threads': 12,
    'targets': wm.CONTROL_TARGETS.copy(),
    'context': wm.CONTEXT.copy(),
    'actions': wm.ACTIONS.copy(),
    'features': wm.FEATURES.copy(),
    'residual_targets': wm.TARGETS.copy()
}

# Derived run facts are recorded once, after the editable settings.
if CONFIG['progress_backend'] not in ['text', 'auto']:
    raise ValueError('Choose text or auto for progress_backend.')
if wm.TQDM_AVAILABLE:
    wm.tqdm = text_tqdm if CONFIG['progress_backend'] == 'text' else tqdm
print('Progress backend:', CONFIG['progress_backend'], '| bars enabled:', CONFIG['progress_bars'])
if CONFIG['training_track'] not in ['full_scalable', 'capped_all_models']:
    raise ValueError('Choose full_scalable or capped_all_models.')
CONFIG['training_windows_per_cultivation'] = (None if CONFIG['training_track'] == 'full_scalable'
                                            else CONFIG['capped_windows_per_cultivation'])
CONFIG['training_backend_sha256'] = wm.file_hash(wm.__file__)
eda_manifest = PROJECT_ROOT / 'dev/dataset_eda/reports/quality_5s_12targets/manifest.json'
CONFIG['eda_reference'] = {'file': str(eda_manifest.relative_to(PROJECT_ROOT)),
                           'sha256': wm.file_hash(eda_manifest) if eda_manifest.is_file() else None,
                           'use': 'Historical descriptive review; no held-out outcomes determine filtering or sampling'}
if CONFIG['input_filter']['mode'] != 'none':
    wp.input_filter(np.zeros((1, 2, len(CONFIG['features']))),
                       CONFIG['input_filter'], CONFIG['features'], SAMPLING_INTERVAL_SECONDS)
for algorithm, settings in CONFIG['neural'].items():
    if settings['batch_size'] < 1 or settings['epochs'] < 1:
        raise ValueError(f'{algorithm}: batch size and epochs must be positive.')
if CONFIG['loader_workers'] != 0:
    raise ValueError('This low-RAM configuration keeps loader_workers=0 to avoid worker dataset duplication.')
for key in ['seed', 'sampling_seed']:
    if isinstance(CONFIG[key], bool) or not isinstance(CONFIG[key], (int, np.integer)) or CONFIG[key] < 0:
        raise ValueError(f'{key} must be a nonnegative integer.')
print('Training seed:', CONFIG['seed'], '| fixed window-selection seed:', CONFIG['sampling_seed'])
DATASET_DIR = PROJECT_ROOT / CONFIG['dataset_path']
MODEL_ROOT = PROJECT_ROOT / CONFIG['model_path']
MODEL_DIR = MODEL_ROOT / CONFIG['run_id']
split_sets = [set(CONFIG['split'][name]) for name in ['train', 'validation', 'test_reserved']]
if any(split_sets[i] & split_sets[j] for i in range(3) for j in range(i + 1, 3)):
    raise ValueError('Cultivations overlap across splits.')
wm.validate_schema(CONFIG)
if not CONFIG['split']['validation']:
    inconsistent = [name for name, settings in CONFIG['neural'].items()
                    if settings.get('selection', 'validation') != 'fixed_epochs']
    if inconsistent:
        raise ValueError(
            'An empty validation split requires selection=fixed_epochs for every neural model; '
            f'still using validation-based early stopping: {inconsistent}.')
    print('All-data run: no validation cultivation, so neural fitting uses a fixed epoch budget '
          'and no held-out score is produced.')
TIMING = wm.time_settings(CONFIG)
CONFIG['history_steps'] = TIMING['history_steps']
CONFIG['observation_operator'] = TIMING['observation_operator']
CONFIG['prediction_horizon_minutes'] = TIMING['sampling_interval_seconds'] / 60
# Convert the anchor duration to steps so it follows the sampling cadence.
CONFIG['baseline_window_steps'] = wm.duration_steps(
    CONFIG.get('baseline_window_minutes', 15), TIMING['sampling_interval_seconds'])
for settings in CONFIG['neural'].values():
    wm.initialization_strategy(settings, CONFIG['target_transform'])
    wm.rollout_settings(settings)
tcn_dilations, receptive_field_steps = [], 1
while receptive_field_steps < TIMING['history_steps']:
    dilation = 2 ** len(tcn_dilations)
    tcn_dilations.append(dilation)
    receptive_field_steps += 4 * dilation
CONFIG['neural']['tcn']['dilations'] = tcn_dilations
CONFIG['neural']['pg_ac_tcn']['dilations'] = tcn_dilations
CUDA_AVAILABLE = wm.torch is not None and wm.torch.cuda.is_available()
CUDA_DEVICE_COUNT = wm.torch.cuda.device_count() if wm.torch is not None else 0
CUDA_UNAVAILABLE_REASON = None
if wm.torch is None:
    CUDA_UNAVAILABLE_REASON = 'PyTorch unavailable: ' + str(wm.IMPORT_ERRORS.get('torch'))
elif not CUDA_AVAILABLE:
    if wm.torch.version.cuda is None:
        CUDA_UNAVAILABLE_REASON = 'This PyTorch installation was built without CUDA.'
    else:
        try:
            wm.torch.cuda.init()
        except Exception as error:
            CUDA_UNAVAILABLE_REASON = f'{type(error).__name__}: {error}'
        else:
            CUDA_UNAVAILABLE_REASON = 'CUDA availability check returned False; inspect this kernel environment.'
DEVICE = ('cuda' if CUDA_AVAILABLE else 'cpu') if CONFIG['device'] == 'auto' else str(CONFIG['device'])
if DEVICE.startswith('cuda') and not CUDA_AVAILABLE:
    raise RuntimeError('CUDA was requested but is unavailable in this kernel. ' + str(CUDA_UNAVAILABLE_REASON))
GPU_NAME = wm.torch.cuda.get_device_name(DEVICE) if DEVICE.startswith('cuda') else None

def show_model_config(algorithm):
    if algorithm not in CONFIG['algorithms']:
        print(f'{algorithm}: not selected; skipping.')
        return
    if algorithm in CONFIG['neural']:
        settings = CONFIG['neural'][algorithm]
        execution_device = DEVICE
        device_name = GPU_NAME if DEVICE.startswith('cuda') else 'CPU'
        device_note = ('PyTorch neural training.' if DEVICE.startswith('cuda') else
                       ('Automatic CPU fallback: ' + str(CUDA_UNAVAILABLE_REASON)
                        if CONFIG['device'] == 'auto' else 'CPU explicitly selected for neural training.'))
    elif algorithm in CONFIG['classical']:
        settings = CONFIG['classical'][algorithm]
        execution_device, device_name = 'cpu', 'CPU'
        device_note = ('Current XGBoost backend uses the CPU histogram method.' if algorithm == 'xgboost'
                       else ('LightGBM gradient-boosted trees run on CPU.' if algorithm == 'lightgbm'
                       else ('HRP-XGB bootstrap ensemble runs on CPU.' if algorithm == 'hrp_xgb'
                             else 'Current scikit-learn estimator runs on CPU.')))
    elif algorithm == 'grey_box':
        settings = CONFIG['physical']
        execution_device, device_name = 'cpu', 'CPU'
        device_note = 'SciPy physical-parameter calibration runs on CPU.'
    else:
        raise ValueError(f'Unknown algorithm: {algorithm}')
    print(f'Model: {algorithm} | configured execution device: {execution_device} | hardware: {device_name}')
    print('Selected:', algorithm in CONFIG['algorithms'], '|', device_note,
          '| fitting remains subject to dependency and memory checks.')
    display({**settings, 'execution_device': execution_device, 'device_name': device_name})

print('Run:', CONFIG['run_id'], '| requested device:', CONFIG['device'], '| neural device:', DEVICE)
print('Kernel Python:', sys.executable)
print('PyTorch:', wm.torch.__version__ if wm.torch is not None else 'unavailable',
      '| compiled CUDA:', wm.torch.version.cuda if wm.torch is not None else 'unavailable')
print('CUDA available in this kernel:', CUDA_AVAILABLE, '| visible CUDA devices:', CUDA_DEVICE_COUNT)
if CUDA_UNAVAILABLE_REASON:
    print('CUDA unavailable reason:', CUDA_UNAVAILABLE_REASON)
    print('GPU visibility depends on the notebook kernel and its execution environment.')
print('History:', CONFIG['history_minutes'], 'minutes /', TIMING['history_steps'], 'samples')
print('Prediction step:', CONFIG['sampling_interval_seconds'], 'seconds | target:', TIMING['observation_operator'])
for algorithm in CONFIG['algorithms']:
    if algorithm in CONFIG['neural']:
        recipe = wm.rollout_settings(CONFIG['neural'][algorithm])
        print(algorithm, '| maximum training rollout:', recipe['steps'], 'steps /',
              recipe['steps'] * SAMPLING_INTERVAL_SECONDS / 60, 'minutes | curriculum:', recipe['curriculum'])
print('Multi-step training keeps gradients through predicted feedback; checkpoint selection remains one-step validation.')
print('TCN receptive field:', receptive_field_steps, 'samples /', receptive_field_steps * SAMPLING_INTERVAL_SECONDS / 60, 'minutes')
display(pd.DataFrame({'target_order': CONFIG['targets'], 'provenance': [TARGET_PROVENANCE[t] for t in CONFIG['targets']]}))
print('Label mode:', LABEL_MODE, '| target transform:', CONFIG['target_transform'])
print('Neural output width:', 2 * len(CONFIG['targets']), '(12 means + 12 log-variances)')
print('History channels:', len(CONFIG['features']), '| measured TSS history included')
print('Input history excludes biomass, BOD, estimated OD/wet/dry mass, calculated lamp lux, and MPPT.')
print('TSS remains a level prediction target even when its history is included; BOD stays independently predicted, not forced to COD/2.')
print('Biomass is an interpolated estimate; no independent biological/compliance validation is claimed.')
if not any(not wm.missing_dependencies(a) for a in CONFIG['algorithms']):
    raise ImportError('None of the selected algorithms has its required dependencies.')

print('Training track:', CONFIG['training_track'])
print('Training sampling:', CONFIG['training_sampling'])
print('Treatment diagnostic cohort:', CONFIG['treatment_diagnostics'], '| excluded from checkpoint selection score')
print('Usable hardware budgets:', CONFIG['ram_budget_gib'], 'GiB RAM /', CONFIG['vram_budget_gib'], 'GiB VRAM')
if DEVICE.startswith('cuda'):
    print('Detected GPU:', GPU_NAME)
    print('Detected VRAM GiB:', round(wm.torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3, 2))
else:
    print('CPU selected. On-demand windows still reduce system RAM; CUDA access must work to use GPU acceleration.')

for algorithm, settings in CONFIG['neural'].items():
    if not 0 <= settings['head_dropout'] < 1 or settings['learning_rate'] <= 0:
        raise ValueError(f'{algorithm}: invalid dropout or learning rate.')
    if settings['scheduler_patience'] >= settings['patience'] or settings['nll_weight'] < 0:
        raise ValueError(f'{algorithm}: scheduler must act before early stopping; NLL weight must be nonnegative.')
print('Training recipe:', CONFIG['training_recipe'])
print('Checkpoint zero predicts persistence for the original nine targets and training means for biomass/BOD/TSS; TSS evaluation also reports observed-history persistence when available.')
print('Derived outputs (value computed from a source target, not learned):', CONFIG['derived_outputs'])
print('Headline targets driving checkpoint selection:', CONFIG['headline_targets'])
derived_rule = [f"{k} = {v['scale']} * {v['source']} + {v['offset']}" for k, v in CONFIG['derived_outputs'].items()]
print('Derivation:', '; '.join(derived_rule) if derived_rule else 'none')
print('Effective point-loss weight bounds:', CONFIG['point_loss_weight_bounds'])
print('Split:', CONFIG['split'])
if not CONFIG['split']['test_reserved']:
    print('No split is held out as test_reserved; every cultivation participates in leave-one-cultivation-out folds.')


In [3]:
# Optional warm-start only from a completed checkpoint with this exact 12-target schema.
# Configure an exact metadata JSON path for each model to continue; this run keeps a new ID.
CONTINUE_TRAINING = False
RESUME_METADATA_BY_ALGORITHM = {
    'gru': None,
    'lstm': None,
    'tcn': None,
    'pg_ac_tcn': None,
    'pets_mlp': None,
    'dlinear': None,
    'nhits': None,
    'tft': None,
    'dnlssm': None,
}
CONFIG['continue_training'] = CONTINUE_TRAINING
CONFIG['resume_metadata_by_algorithm'] = RESUME_METADATA_BY_ALGORITHM
selected_neural = set(CONFIG['algorithms']) & set(RESUME_METADATA_BY_ALGORITHM)
resume_paths = {name: path for name, path in RESUME_METADATA_BY_ALGORITHM.items()
                if path is not None and str(path).strip()}
if CONTINUE_TRAINING:
    if not resume_paths:
        raise ValueError('Continue mode is enabled but no resume metadata path is configured.')
    unknown = set(resume_paths) - selected_neural
    if unknown:
        raise ValueError(f'Resume paths configured for algorithms not selected in this run: {sorted(unknown)}')
elif resume_paths:
    print('Resume paths are configured but ignored because CONTINUE_TRAINING=False.')
print('Continue training:', CONTINUE_TRAINING,
      '| source artifacts:', {name: str(path) for name, path in resume_paths.items()} if CONTINUE_TRAINING else 'none')
# Full-state recovery is separate from the weight-only warm-start option above.
# Resolve and validate recovery paths already configured in CONFIG.
recovery_sources = {}
for algorithm, configured_path in CONFIG['recovery']['resume_by_algorithm'].items():
    if configured_path:
        path = Path(configured_path).expanduser()
        recovery_sources[algorithm] = str(path if path.is_absolute() else PROJECT_ROOT / path)
if set(recovery_sources) - set(CONFIG['algorithms']):
    raise ValueError('Recovery sources must belong to selected algorithms.')
if CONTINUE_TRAINING and recovery_sources:
    raise ValueError('Choose full-state recovery or weight-only warm start, not both.')
CONFIG['recovery']['resume_by_algorithm'] = recovery_sources
CONFIG['recovery']['directory'] = str(MODEL_DIR / 'recovery')
print('Recovery:', CONFIG['recovery'])
print('Recovery saves at epoch boundaries; an interrupted unfinished epoch is replayed.')


Continue training: False | source artifacts: none
Recovery: {'enabled': True, 'every_epochs': 5, 'directory': '/home/rahmadbisma/Documents/ITS/MBRL_MicroAlgae/dev/world_model/model/5S_BT128_SD44_ROUT1_20261109T221542509767Z/recovery', 'resume_by_algorithm': {}}
Recovery saves at epoch boundaries; an interrupted unfinished epoch is replayed.


## ***Data Load***

In [ ]:
train_frames, train_sources = wm.load_frames(DATASET_DIR, CONFIG['split']['train'], CONFIG)
validation_frames, validation_sources = wm.load_frames(DATASET_DIR, CONFIG['split']['validation'], CONFIG)
display(pd.concat([train_sources.assign(split='train'), validation_sources.assign(split='validation')], ignore_index=True))
source_manifest = pd.concat([train_sources.assign(split='train'), validation_sources.assign(split='validation')], ignore_index=True).to_dict('records')
print('Fitting cultivations:', CONFIG['split']['train'], '| selection cultivation:', CONFIG['split']['validation'])

CONFIG['source_manifest'] = source_manifest


## ***Preprocessing***

### ***Data-quality review, representative starts and transformations***

**EDA assessment:** five-second priority-channel variation is mostly slower than a minute, but COD/BOD have short spike candidates and pH/ammonia have long repeated-value runs. Light requires separate attention. These observations support preserving a raw baseline and reviewing events, rather than blanket denoising. Variance describes variability and does not identify measurement noise by itself.

Retrospective quality flags are only descriptive. For example, a jump that returns immediately may be flagged as a spike, while an unchanged pH value for many minutes may reflect quantization or a frozen sensor. Flags may inspect neighbouring observations and therefore never enter model inputs, sampling decisions or automatic cleaning. A zero robust scale makes the outlier screen unavailable rather than proving the signal clean.

Complete histories require finite inputs, valid timestamp continuity and available source-quality flags. Minute averages also require complete source-bin coverage. Neither histories nor filters bridge gaps or cultivation boundaries. Logged dose volumes are retained as volumes; settings are not silently converted into delivered energy.

Capped training reserves a declared part of its budget for positive logged doses and changes in aeration or lamp setting, then fills the remainder across cultivation time. For example, rare dose-active prediction timestamps no longer depend on chance inclusion in a time sample. Each available treatment channel is considered; absent events are reported rather than invented. All learned models share the same starts. This deliberately enriched training distribution is recorded and is different from representative held-out assessment.

Optional wider spacing changes the background forecast starts, not the observations within a history. For example, a model can receive five-second history readings while training starts are thirty seconds apart. Eligible treatment events bypass the background-spacing restriction. The preview compares candidate counts without training or choosing a winner. The default retains native start spacing.

Checkpoint validation remains a representative time sample. A separate treatment audit contains logged events and their declared follow-up periods. For example, an audit may inspect the first minutes after dosing without assuming that this is the true biological response delay. Its errors are reported during neural training and candidate searches, but do not enter the loss, scheduler or checkpoint-selection score.

Scaling is fitted on training histories only. For example, a training mean and scale are reused unchanged on held-out measurements. The optional trailing median is applied within each available history, resets at its beginning and leaves labels, treatment settings and the latest-reading reference unchanged. If enabled, training scaling uses those filtered histories too. The same observation rule is reused during recursive evaluation.

The original nine process targets are predicted as changes from their latest raw reading. Biomass, BOD and TSS remain level targets. This assignment is explicit and independent of the input schema: adding TSS history must not silently switch its prediction to a delta. Reference forecasts use observed-history persistence when available (including TSS in experiment B) and training means for output-only targets. A combined reference is therefore a hybrid, not pure persistence.

Some run facts are calculated rather than entered twice. For example, choosing five-second sampling also determines the one-step prediction duration; choosing a capped track determines the training-window limit. The temporal convolution spacing expands automatically to cover the supplied history. Candidate lists are the single source of truth for Ridge and Gaussian-process searches. Reference initialization chooses persistence or a training mean according to the target formulation; random initialization is a separate alternative.

Training-only output-dependency checks describe near-affine relationships such as a TSS column tracking turbidity. Correlation alone does not establish an independent sensor or a causal relationship. All outputs remain, and the full-target score is explicitly an average over potentially dependent columns. Light is reviewed by clock day/night and lamp-setting changes; hourly display summaries are diagnostic plots only and never replace training observations.

In [ ]:
def eligible_for_diagnostics(frame):
    eligible = wm.valid_anchors(frame, TIMING['history_steps'], SAMPLING_INTERVAL_SECONDS, CONFIG['features'])
    return eligible[np.isfinite(frame[CONFIG['targets']].to_numpy()[eligible]).all(axis=1)]

sampling_config = {**CONFIG, 'seed': CONFIG['sampling_seed']}
window_index = wm.WindowIndex(train_frames, sampling_config, CONFIG['training_windows_per_cultivation'])
# An all-data final model has no held-out cultivation, so training uses a fixed epoch
# budget and this cell keeps the cohort empty instead of inventing one.
if not validation_frames:
    print('No validation cultivation: training uses selection=fixed_epochs; no held-out score will be reported.')
raw_validation, validation_window_counts = (
    wm.raw_windows(validation_frames, sampling_config, CONFIG['validation_windows_per_cultivation'])
    if validation_frames else (None, pd.DataFrame()))
treatment_anchors, treatment_validation_counts = (
    wp.treatment_audit_starts(validation_frames, sampling_config, wm.ACTIONS, eligible_for_diagnostics)
    if validation_frames else ({}, pd.DataFrame()))
if sum(len(v) for v in treatment_anchors.values()):
    raw_treatment_validation, _ = wm.raw_windows(validation_frames, sampling_config, anchor_override=treatment_anchors)
else:
    raw_treatment_validation = None
tss_review = []
for split, frames in [('train', train_frames), ('validation', validation_frames)]:
    for cultivation, frame in frames.items():
        values = frame['1_tss(mg/L)'].to_numpy()
        tss_review.append({'split': split, 'cultivation': cultivation, 'rows': len(values),
                           'finite_readings': int(np.isfinite(values).sum()),
                           'negative_readings': int((np.isfinite(values) & (values < 0)).sum()),
                           'source': 'Actual CSV TSS; README documents dedicated newer-hardware sensor'})
display(Markdown('#### ***TSS source review; measured history is used unchanged, the output is derived from turbidity***'))
display(pd.DataFrame(tss_review))
preprocessing = wm.fit_preprocessing(window_index, train_frames)
validation = wm.transform(raw_validation, preprocessing) if raw_validation is not None else {}
raw_validation = validation['raw'] if raw_validation is not None else None
if raw_treatment_validation is not None:
    validation['treatment_diagnostic'] = wm.transform(raw_treatment_validation, preprocessing)
quality_review = pd.DataFrame([{'cultivation': k, **row} for k, frame in {**train_frames, **validation_frames}.items()
                               for row in frame.attrs.get('quality_summary', [])])
display(Markdown('#### ***Observed quality screens; no automatic deletion***'))
display(quality_review)
CONFIG['quality_summary'] = quality_review.to_dict('records')
training_selections = {k: window_index.entries[window_index.entries[:,0] == k,1] for k in train_frames}
treatment_training_coverage = wp.treatment_coverage(train_frames, training_selections, sampling_config, wm.ACTIONS, eligible_for_diagnostics)
spacing_preview = wp.training_spacing_preview(train_frames, sampling_config, wm.ACTIONS, eligible_for_diagnostics)
output_dependencies = wp.output_dependency_report(train_frames, CONFIG['output_dependency_pairs'])
light_review, light_hourly = wp.light_regime_report(train_frames, SAMPLING_INTERVAL_SECONDS)
CONFIG['output_dependency_review'] = output_dependencies.to_dict('records')
CONFIG['training_treatment_coverage'] = treatment_training_coverage.to_dict('records')
CONFIG['light_regime_review'] = light_review.replace({np.nan: None}).to_dict('records')
for title, table in [('Treatment coverage in the shared training cohort', treatment_training_coverage),
                     ('Independent treatment-validation coverage', treatment_validation_counts),
                     ('Spacing experiment preview; counts only', spacing_preview),
                     ('Training-only dependencies; outputs remain unchanged', output_dependencies),
                     ('Training-only light regimes; clock day/night is approximate', light_review)]:
    display(Markdown(f'#### ***{title}***'))
    display(table)
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(len(light_hourly), 1, figsize=(13, 3 * len(light_hourly)), squeeze=False)
    for ax, (k, table) in zip(axes.flat, light_hourly.items()):
        ax.plot(table.index, table.light_median, color='#186A8C', label='Hourly light median')
        ax.fill_between(table.index, table.light_p10, table.light_p90, color='#186A8C', alpha=.2, label='10th–90th percentile')
        ax.set_ylabel('Measured light (lux)'); ax.set_title(f'K{k:02d}: light variability and recorded lamp setting')
        other = ax.twinx(); other.plot(table.index, table.pwm_mean, color='#C67414', alpha=.8, label='Hourly mean lamp PWM')
        other.set_ylabel('Lamp PWM (%)')
        ax.legend(loc='upper left', fontsize=8); other.legend(loc='upper right', fontsize=8)
        ax.grid(alpha=.2)
    fig.suptitle('Display summaries only; native training observations remain unchanged')
    fig.tight_layout(rect=(0, 0, 1, .96))
    buffer = BytesIO(); fig.savefig(buffer, format='png', dpi=100, bbox_inches='tight')
    display(Image(data=buffer.getvalue())); plt.close(fig)
print('Reference:', preprocessing['baseline_description'])
display(window_index.counts)
display(validation_window_counts)
print('Training windows:', len(window_index))
for algorithm in CONFIG['algorithms']:
    if algorithm in CONFIG['neural']:
        steps = CONFIG['neural'][algorithm]['rollout_steps']
        eligible = wm.rollout_training_positions(window_index, steps)
        print(algorithm, '| complete training-rollout starts:', len(eligible), '/', len(window_index),
              '| excluded:', len(window_index) - len(eligible))
print('Source feature arrays MiB:', round(sum(store['values'].nbytes for store in window_index.stores.values()) / 1024**2, 2))
print('Neural batches per selected-cohort epoch:', int(np.ceil(len(window_index) / CONFIG['neural']['gru']['batch_size'])))
print('Dense GPR kernel matrix alone, GiB:', round(len(window_index)**2 * 8 / 1024**3, 2))
print('No expanded training history tensor is allocated.')
RESULTS = []
METADATA_PATHS = []
TABULAR_CACHE = {}

def get_tabular_train():
    if 'prepared' not in TABULAR_CACHE:
        directory = PROJECT_ROOT / CONFIG['cache_path'] / CONFIG['run_id']
        TABULAR_CACHE['prepared'] = wm.prepare_tabular(window_index, preprocessing, CONFIG, directory)
        print('Estimated classical working arrays GiB:', round(TABULAR_CACHE['prepared']['estimated_working_arrays_gib'], 2))
    return TABULAR_CACHE['prepared']

def validate_fitted(fitted):
    if not wm.validation_available(validation):
        # Final all-data model: every cultivation was used for fitting, so there is
        # no held-out cohort to score. Monitoring metrics were already recorded per
        # epoch; report that explicitly instead of inventing a score.
        print('No held-out validation cohort: this model is fitted on all available cultivations.')
        print('Report performance from the leave-one-cultivation-out folds, not from this run.')
        empty = pd.DataFrame(columns=['target', 'MAE', 'RMSE', 'NRMSE', 'baseline_method',
                                      'baseline_RMSE', 'baseline_skill', 'prediction_status'])
        empty.attrs['macro_NRMSE_headline'] = None
        empty.attrs['macro_NRMSE_all'] = None
        empty.attrs['headline_targets'] = wm.headline_target_names(preprocessing)
        fitted['validation_evaluated'] = False
        return empty, np.nan
    predicted, distribution = wm.predict(fitted, validation)
    if not np.isfinite(predicted).all():
        raise FloatingPointError('Non-finite validation prediction; artifact not marked completed.')
    columns = fitted.get('targets', CONFIG['targets'])
    metrics, score = wm.point_metrics(raw_validation['truth'], predicted, raw_validation['latest'], preprocessing, columns)
    display(metrics[['target', 'MAE', 'RMSE', 'NRMSE', 'baseline_method', 'baseline_RMSE', 'baseline_skill', 'prediction_status']])
    priority = [t for t in CONFIG['priority_targets'] if t in columns]
    display(Markdown('#### ***Priority-channel validation***'))
    display(metrics.loc[metrics.target.isin(priority)])
    print('Priority summary:', wp.priority_summary(metrics, priority) if priority else 'Not applicable to this target subset')
    event_metrics, event_summary = wm.treatment_validation_diagnostic(fitted, validation)
    if event_metrics is not None:
        display(Markdown('#### ***Separate treatment validation; diagnostic only***'))
        display(event_metrics)
        print(event_summary)
        fitted['validation_treatment_metrics'] = event_metrics.to_dict('records')
    interval = CONFIG['sampling_interval_seconds']
    validity = pd.DataFrame(wd.validity_rows(predicted, columns, 'one_step_checkpoint', 1, interval))
    response = coverage = recursive_validity = pd.DataFrame()
    if columns == CONFIG['targets']:
        response, coverage, recursive_validity = wd.event_response_diagnostics(wm, fitted, validation_frames, CONFIG)
    display(Markdown('#### ***Physical validity by channel and recursive step; diagnostic only***'))
    display(validity)
    if not recursive_validity.empty:
        display(recursive_validity.groupby(['cohort', 'target'], as_index=False).agg(
            negative=('negative', 'sum'), nonfinite=('nonfinite', 'sum'),
            ph_outside_0_14=('ph_outside_0_14', 'sum'), minimum_prediction=('minimum_prediction', 'min'),
            maximum_negative_magnitude=('maximum_negative_magnitude', 'max')))
    if not coverage.empty:
        display(Markdown('#### ***Event/no-event recursive validation coverage***'))
        display(coverage)
    if not response.empty:
        display(Markdown('#### ***Response direction, amplitude and delay; logged-action association only***'))
        display(response.groupby(['group', 'target', 'horizon_minutes'], as_index=False).agg(
            origins=('origin_index', 'size'), trajectory_RMSE=('trajectory_RMSE', 'mean'),
            persistence_RMSE=('persistence_trajectory_RMSE', 'mean'),
            direction_agreement=('direction_correct', 'mean'),
            actual_amplitude=('actual_response_amplitude', 'mean'),
            predicted_amplitude=('predicted_response_amplitude', 'mean'),
            actual_delay_seconds=('actual_response_delay_seconds', 'mean'),
            predicted_delay_seconds=('predicted_response_delay_seconds', 'mean')))
    fitted['validation_event_response'] = response.to_dict('records')
    fitted['validation_event_coverage'] = coverage.to_dict('records')
    # Whether this model may be handed to an RL policy at all, and why not.
    fitted['action_response_gate'] = wd.action_response_gate(response, coverage, CONFIG)
    fitted['action_coverage_limitations'] = wd.action_coverage_limitations(coverage)
    gate = fitted['action_response_gate']
    display(Markdown('#### ***Action-response gate; the RL usability check***'))
    display(pd.DataFrame(gate['per_action']) if gate['per_action'] else pd.DataFrame())
    print('RL eligible:', gate['rl_eligible'], '| status:', gate['gate_status'])
    print('Control (no-event) direction agreement:', gate.get('control_direction_agreement'))
    print('Reason:', gate['reason'])
    # An uncertainty penalty is only useful if uncertainty separates seen from unseen.
    fitted['uncertainty_discrimination'] = wm.uncertainty_discrimination(
        fitted, validation, validation, preprocessing) if fitted.get('uncertainty_mode') else {
        'status': 'no_model_uncertainty'}
    print('Uncertainty discrimination:', fitted['uncertainty_discrimination']['status'])
    fitted['validation_prediction_validity'] = pd.concat([validity, recursive_validity], ignore_index=True).to_dict('records')
    fitted['validation_evaluated'] = True
    print('Validation macro NRMSE (headline targets):', score, '| fit seconds:', round(fitted['fit_seconds'], 3))
    print('Validation macro NRMSE (all targets):', metrics.attrs.get('macro_NRMSE_all'),
          '| headline targets scored:', len(metrics.attrs.get('headline_targets', [])))
    derived = metrics.attrs.get('headline_targets', [])
    print('Headline set:', ', '.join(derived) if derived else 'not declared')
    return metrics, score

print('Classical memory policy:', wm.tabular_skip_reason(len(window_index), CONFIG) or 'within configured working-array budget')
print(f'For an eight-family comparison at {SAMPLING_INTERVAL_SECONDS}-second cadence, use capped_all_models; '
      'no algorithm receives a hidden smaller sample.')


In [6]:
RESUME_STATE_DICTS = {}
RESUME_SOURCES = {}
if CONFIG['continue_training']:
    for algorithm, configured_path in CONFIG['resume_metadata_by_algorithm'].items():
        if configured_path is None or not str(configured_path).strip():
            continue
        metadata_path = Path(configured_path).expanduser()
        if not metadata_path.is_absolute():
            metadata_path = PROJECT_ROOT / metadata_path
        states, source = wm.load_resume_checkpoint(metadata_path, algorithm, CONFIG, preprocessing)
        RESUME_STATE_DICTS[algorithm] = states
        RESUME_SOURCES[algorithm] = source
        print(f"Warm-start ready: {algorithm} from {source['artifact']} ({len(states)} member(s)); optimizer state will be fresh.")

def train_neural_configured(algorithm):
    if algorithm not in CONFIG['algorithms']:
        raise ValueError(f'{algorithm}: not selected in CONFIG[algorithms].')
    return wm.train_neural(
        algorithm, window_index, validation, preprocessing, CONFIG, DEVICE,
        resume_state_dicts=RESUME_STATE_DICTS.get(algorithm),
        resume_source=RESUME_SOURCES.get(algorithm),
        # Lets checkpoint selection follow CONFIG rollout_steps with a matching
        # recursive validation, instead of always scoring one step.
        validation_frames=(globals().get('validation_frames') or None))

## ***Deep Learning***

Each neural model predicts both a mean and uncertainty for each target. For example, an oxygen forecast may say a small increase is expected while assigning uncertainty to that increase. Original historical targets use predicted changes; added output-only targets use predicted levels.

The objective combines normalized squared error with a Gaussian likelihood component. The normalization allows channels with different units to contribute sensibly, but does not make every biological outcome equally trustworthy. Separate priority-channel reports expose errors that an aggregate can conceal. No additional priority weights are assumed.

Dropout, weight decay, gradient clipping and a validation-driven learning-rate schedule regularize fitting. The configured epoch limit is a ceiling; early stopping can end training sooner. Checkpoint selection uses complete-target validation error and rejects non-finite forecasts. Starting predictions use persistence or training means as appropriate. If the starting checkpoint wins, the report explicitly states that training did not improve it. The displayed runtime configuration and saved metadata record exact settings.

For multi-step training, each batch contains an initial real history, future logged actions/clock, and a sequence of target labels. Predicted means replace observed sensor channels in the next history. Unpredicted context persists, biomass and BOD labels never enter history; TSS is fed back only from predictions in the required history, and future sensor readings are never used as inputs. Gradients pass through the complete rollout. The loss averages normalized point error and likelihood across steps, with optional discounting. Histories are causally filtered again at each step using the same rule as evaluation.

Selected starts near gaps, incomplete readings, missing targets, or episode ends are excluded if they cannot support the full configured rollout. Preprocessing and the original selected pool remain unchanged; eligible counts and excluded pool positions are recorded for each neural model. Classical/physical fitting is unchanged. Training curves record the active rollout length, and metadata stores the curriculum, feedback policy and peak CUDA allocated memory when available.

Checkpoint selection, scheduler and early stopping still use the same **one-step validation macro NRMSE**. A checkpoint selected during an early curriculum stage is labeled with that stage; completing later stages does not imply that they improved validation. Use `evaluation.ipynb` for independent 1/5/15/60-minute rollout assessment. Changing the training rollout does not change the saved model's one-step inference contract.


### ***GRU***

A recurrent model updates a hidden summary as each observation arrives. Gates decide what to retain and what to replace. For example, recent dosing and a slowly changing oxygen trend can both affect the next forecast.

The following code constructs the model, fits it on the shared training cohort, validates it and saves the matching artifact and metadata.

In [7]:
ALGORITHM = 'gru'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

gru: not selected; skipping.


### ***LSTM***

A recurrent model maintains a cell memory alongside its hidden state. Separate gates control writing, forgetting and reading. For example, it may retain a slow treatment response while reacting to a recent change.

The following code constructs the model, fits it on the shared training cohort, validates it and saves the matching artifact and metadata.

In [8]:
ALGORITHM = 'lstm'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

lstm: not selected; skipping.


### ***Causal TCN***

Dilated causal convolutions look backward at increasingly spaced observations. For example, successive layers can combine recent seconds with older trends. Future measurements never enter a forecast; the effective history remains limited by the available input window.

The following code constructs the model, fits it on the shared training cohort, validates it and saves the matching artifact and metadata.

In [9]:
ALGORITHM = 'tcn'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

tcn: not selected; skipping.


### ***PG-AC-TCN: physics-guided action-conditioned probabilistic TCN***

Uses the existing 21-channel history: the 16 sensor/context channels feed a causal TCN, while the five historical actions feed a temporal action encoder. The current five-action proposal and three clock fields enter a FiLM-style gated fusion block before the 12-output probabilistic transition head. Outputs follow the existing mixed residual/level target transforms, including independent BOD and biomass proxy levels and recursive TSS level feedback.

Physics penalties are configurable but disabled by default (`physics_loss_weight=0`, `physics_bounds={}`). Enable them only for explicit, validated target bounds; the training helper rejects a nonzero weight without supplied bounds. No unverified kinetic or mass-balance assumption is imposed. The current data records logged treatments, so this remains conditional action replay rather than proof of counterfactual action effects.


In [ ]:
ALGORITHM = 'pg_ac_tcn'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

### ***Probabilistic MLP ensemble***

Each member maps the flattened history and treatment context to means and variances. Members receive block-bootstrap training samples. For example, disagreement between members can accompany limited training support, but does not by itself prove calibrated uncertainty. This is a history-adapted PETS-style predictor, not the complete PETS controller.

The following code constructs the model, fits it on the shared training cohort, validates it and saves the matching artifact and metadata.

In [11]:
ALGORITHM = 'pets_mlp'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

pets_mlp: not selected; skipping.


### ***MOPO: model-based offline policy optimization (Yu et al., NeurIPS 2020).***

A probabilistic network ensemble whose uncertainty is the maximum predicted standard deviation across members, `u = max_i ||sigma_i||`. That is the pessimistic error estimate MOPO uses to penalise rewards, `r_tilde = r_hat - lambda * u`, in its uncertainty-penalised MDP. The reward penalty and the SAC policy optimizer act on a reward function and an actor-critic; those belong in `dev/rl_controller`, so this stage provides the dynamics model and its uncertainty estimate only.

All members share one trunk and are trained independently by maximum likelihood. Uncertainty is reported per target and target-unit aware; it is a model-error estimate, not a calibrated confidence guarantee. With fewer than two members there is no disagreement to measure, so the helper rejects that configuration.


In [ ]:
ALGORITHM = 'mopo_ensemble'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

### ***MOUP: model-based offline RL with uncertainty estimation and policy constraint (Zhu et al., IEEE TAI 2024).***

Ensemble dropout networks. Each member keeps dropout active at inference and is evaluated with several stochastic passes, giving `Sigma_hat = Var[dropout] + mean predictive variance`. This fuses the epistemic uncertainty that plain bootstrap ensembles estimate unreliably with the aleatoric noise of the probabilistic head. The paper's MMD state-mismatch constraint is a policy objective and is not part of the dynamics model.

All members share one trunk and are trained independently by maximum likelihood. Uncertainty is reported per target and target-unit aware; it is a model-error estimate, not a calibrated confidence guarantee. With fewer than two members there is no disagreement to measure, so the helper rejects that configuration.


In [ ]:
ALGORITHM = 'moup_mlp'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

### ***DLinear action-conditioned linear dynamics***

Inspired by Zeng et al. (AAAI 2023, DOI: 10.1609/aaai.v37i9.26317). Decomposes the shared sensor history into moving-average trend and residual components. A learned action branch conditions the one-step transition; recursive evaluation uses the common logged-action replay contract. This extends the original channel-independent forecaster for controlled dynamics.


In [12]:
ALGORITHM = 'dlinear'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

dlinear: not selected; skipping.


### ***N-HiTS action-conditioned multiscale dynamics***

Inspired by Challu et al. (AAAI 2023, DOI: 10.1609/aaai.v37i6.25854). Pooled MLP blocks operate at multiple temporal scales and emit backcasts and transition forecasts. The forecasts are adapted to the common action-conditioned one-step world-model interface and are recursively evaluated; this is not a reproduction of the paper’s complete multi-horizon system.


In [13]:
ALGORITHM = 'nhits'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

nhits: not selected; skipping.


### ***Temporal Fusion Transformer (TFT)***

Inspired by Lim et al. (International Journal of Forecasting, 2021, DOI: 10.1016/j.ijforecast.2021.03.012). Variable selection, recurrent temporal encoding, and attention produce an action-conditioned transition distribution. The current action and clock covariates enter the transition head; this is a compact adaptation, not a reproduction of every original TFT component.


In [14]:
ALGORITHM = 'tft'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

tft: not selected; skipping.


### ***Deep nonlinear state-space model (DNLSSM)***

Inspired by Du et al. (CAAI Transactions on Intelligence Technology, 2023 issue, DOI: 10.1049/cit2.12085). An LSTM encodes sensor/action history into a latent Gaussian belief. An action-conditioned nonlinear latent transition and neural observation model are propagated with a diagonal Unscented Kalman Filter (UKF) sigma-point calculation to produce next-state means and predictive variances. As new predicted observations enter recursive history, the encoder updates the latent belief. This is an adapted diagonal-covariance implementation; the paper’s complete parameter-estimation and covariance treatment is not reproduced.


In [15]:
ALGORITHM = 'dnlssm'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'not_selected'})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'skipped', 'reason': str(reason)})
    else:
        display(wm.make_architecture(ALGORITHM, CONFIG))
        try:
            fitted = train_neural_configured(ALGORITHM)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            for checkpoint in fitted['selected_checkpoints']:
                if checkpoint.get('warm_start_selected'):
                    checkpoint_label = ' (resumed checkpoint; additional training did not improve validation)'
                elif checkpoint['initialization_selected']:
                    checkpoint_label = ' (initialization; training did not improve validation)'
                else:
                    checkpoint_label = ''
                print(f"{ALGORITHM} member {checkpoint['member'] + 1} | Best epoch: {checkpoint['selected_epoch']}"
                      f" | validation macro NRMSE: {checkpoint['validation_macro_NRMSE']:.6f}"
                      f" | epochs run: {checkpoint['epochs_run']}{checkpoint_label}")
            print('Training rollout:', fitted['training_rollout'])
            print('Peak CUDA allocated GiB:', fitted['peak_cuda_allocated_gib'])
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'neural', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})

dnlssm: not selected; skipping.


## ***Machine Learning***

### ***Ridge ARX***

A linear predictor relates historical readings and external treatments to the next output. Ridge regularization shrinks coefficients to reduce unstable fits when inputs are highly correlated. For example, it can learn a small oxygen change from past oxygen and aeration.

The following code constructs the model, fits it on the shared training cohort, validates it and saves the matching artifact and metadata.

In [16]:
ALGORITHM = 'ridge'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'not_selected'})
elif wm.tabular_skip_reason(len(window_index), CONFIG):
    reason = wm.tabular_skip_reason(len(window_index), CONFIG)
    print(reason)
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped_memory_policy', 'reason': reason})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped', 'reason': str(reason)})
    else:
        try:
            fitted = wm.train_classical(ALGORITHM, get_tabular_train(), validation, preprocessing, CONFIG)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})


ridge: not selected; skipping.


### ***Random Forest***

Many regression trees average their predictions. Each tree partitions examples using feature thresholds, allowing nonlinear relationships. For example, response can differ at low versus high oxygen. Tree spread is not automatically a calibrated prediction interval.

The following code constructs the model, fits it on the shared training cohort, validates it and saves the matching artifact and metadata.

In [17]:
ALGORITHM = 'random_forest'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'not_selected'})
elif wm.tabular_skip_reason(len(window_index), CONFIG):
    reason = wm.tabular_skip_reason(len(window_index), CONFIG)
    print(reason)
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped_memory_policy', 'reason': reason})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped', 'reason': str(reason)})
    else:
        try:
            fitted = wm.train_classical(ALGORITHM, get_tabular_train(), validation, preprocessing, CONFIG)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})


random_forest: not selected; skipping.


### ***XGBoost***

Boosted trees are added sequentially to correct remaining prediction errors. Regularization and validation stopping limit complexity. For example, later trees can refine a treatment-response pattern missed by earlier trees.

The following code constructs the model, fits it on the shared training cohort, validates it and saves the matching artifact and metadata.

In [18]:
ALGORITHM = 'xgboost'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'not_selected'})
elif wm.tabular_skip_reason(len(window_index), CONFIG):
    reason = wm.tabular_skip_reason(len(window_index), CONFIG)
    print(reason)
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped_memory_policy', 'reason': reason})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped', 'reason': str(reason)})
    else:
        try:
            fitted = wm.train_classical(ALGORITHM, get_tabular_train(), validation, preprocessing, CONFIG)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})


xgboost: not selected; skipping.


### ***HRP-XGB: hybrid residual probabilistic XGBoost ensemble***

Uses the existing lagged 21-feature history, the proposed current five-action vector and clock fields. The available mechanistic reference is the existing transition baseline: persistence for targets present in history and the training mean for output-only BOD/biomass labels. Per-target XGBoost models learn normalized residual corrections to that baseline; they are not presented as a calibrated biological kinetics model.

Five ensemble members bootstrap whole training cultivations. Predictive variance combines member disagreement with target-wise training out-of-bootstrap residual MSE. With only three training cultivations, this uncertainty estimate is provisional and may be unstable. The existing XGBoost and physical grey-box baselines remain separately selectable. Recursive evaluation uses the common action replay and target feedback rules.


In [19]:
ALGORITHM = 'hrp_xgb'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'not_selected'})
elif wm.tabular_skip_reason(len(window_index), CONFIG):
    reason = wm.tabular_skip_reason(len(window_index), CONFIG)
    print(reason)
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped_memory_policy', 'reason': reason})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped', 'reason': str(reason)})
    else:
        try:
            fitted = wm.train_classical(ALGORITHM, get_tabular_train(), validation, preprocessing, CONFIG)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})


hrp_xgb: not selected; skipping.


### ***LightGBM autoregressive dynamics***

Adapted from In and Jung (International Journal of Forecasting, 2022, DOI: 10.1016/j.ijforecast.2021.11.007). LightGBM gradient-boosted trees learn each target transition from flattened lagged sensor/action history, the proposed current action, and clock features. Recursive forecasts feed simulated target values back through the shared lag window; tree extrapolation remains limited to logged support.


In [20]:
ALGORITHM = 'lightgbm'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'not_selected'})
elif wm.tabular_skip_reason(len(window_index), CONFIG):
    reason = wm.tabular_skip_reason(len(window_index), CONFIG)
    print(reason)
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped_memory_policy', 'reason': reason})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped', 'reason': str(reason)})
    else:
        try:
            fitted = wm.train_classical(ALGORITHM, get_tabular_train(), validation, preprocessing, CONFIG)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})


lightgbm: not selected; skipping.


### ***Gaussian Process Regression***

A kernel expresses similarity between histories, and similar training cases inform a new prediction. It also produces predictive uncertainty. For example, a history far from training support may be less certain. Dense fitting is restricted by its quadratic memory needs.

The following code constructs the model, fits it on the shared training cohort, validates it and saves the matching artifact and metadata.

In [21]:
ALGORITHM = 'gpr'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'not_selected'})
elif wm.gpr_skip_reason(len(window_index), CONFIG):
    reason = wm.gpr_skip_reason(len(window_index), CONFIG)
    print(reason)
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped_memory_policy', 'reason': reason})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'skipped', 'reason': str(reason)})
    else:
        try:
            fitted = wm.train_classical(ALGORITHM, get_tabular_train(), validation, preprocessing, CONFIG)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'classical', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})


gpr: not selected; skipping.


## ***Physical Baselines***

### ***Reduced oxygen and water-temperature grey-box model***

A grey-box model combines physical relationships with parameters estimated from data. Oxygen follows effective aeration exchange, light-related production and respiration; water temperature follows exchange with ambient air plus lamp heating and an effective bias. For example, increasing aeration changes the estimated oxygen exchange rate, while changing lamp setting can affect both light and heat.

Calibration uses training data only. Integration follows the saved observation duration: a point-reading experiment predicts an endpoint, while an average-reading experiment predicts an interval mean. Approximating the previous boundary state from an average is a limitation. Recorded light and ambient inputs are used only when available; future measured light is not supplied.

This reduced model predicts only oxygen and water temperature. It does not invent pump effects or complete reactor physics. Its matched two-target comparison stays separate from the full learned-model ranking. Longer recursive evaluation requires a defensible light/background state model and is not claimed here.

In [22]:
ALGORITHM = 'grey_box'
show_model_config(ALGORITHM)
if ALGORITHM not in CONFIG['algorithms']:
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'physical', 'status': 'not_selected'})
elif not {'1_do(mg/L)', '1_temp_air(C)'}.issubset(CONFIG['targets']):
    reason = 'Reduced grey-box requires DO and temperature; it does not implement the selected outcome targets.'
    print('Skipped:', ALGORITHM, reason)
    RESULTS.append({'algorithm': ALGORITHM, 'category': 'physical', 'status': 'skipped', 'reason': reason})
else:
    missing = wm.missing_dependencies(ALGORITHM)
    if missing:
        reason = {name: wm.IMPORT_ERRORS.get(name) for name in missing}
        print('Skipped:', ALGORITHM, reason)
        RESULTS.append({'algorithm': ALGORITHM, 'category': 'physical', 'status': 'skipped', 'reason': str(reason)})
    else:
        try:
            fitted = wm.train_physical(window_index, validation, preprocessing, CONFIG)
            print(f"{ALGORITHM} | actual fit device: {fitted['device']}")
            validation_metrics, score = validate_fitted(fitted)
            metadata_path = wm.save_model(fitted, MODEL_DIR, CONFIG, preprocessing, source_manifest,
                                               window_index, validation, validation_metrics)
            METADATA_PATHS.append(metadata_path)
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'physical', 'status': 'completed',
                            'validation_macro_NRMSE': score, 'fit_seconds': fitted['fit_seconds'],
                            'metadata': metadata_path.name})
            print('Saved:', metadata_path.name)
        except Exception as error:
            traceback.print_exc()
            RESULTS.append({'algorithm': ALGORITHM, 'category': 'physical', 'status': 'failed',
                            'reason': f'{type(error).__name__}: {error}'})


grey_box: not selected; skipping.


### ***Run record***

A run record identifies completed, skipped and failed models and their artifact files. For example, a model that exceeds the memory budget remains visible as skipped rather than receiving a fabricated score. Cross-model comparisons are produced only after reloading artifacts in the evaluation notebook.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = MODEL_DIR / f"run_manifest_{CONFIG['prefix']}_{CONFIG['run_id']}.json"
manifest_path.write_text(json.dumps(wm.json_safe({
    'config': CONFIG, 'results': RESULTS,
    'model_metadata_files': [p.name for p in METADATA_PATHS]}), indent=2, allow_nan=False), encoding='utf-8')
print('Saved model/metadata pairs:', len(METADATA_PATHS))
print('Explicit evaluation run ID:', CONFIG['run_id'])
print('Manifest:', manifest_path.name)
print('Run evaluation.ipynb for the model comparison, graphs, and comparison CSV exports.')
print('Split used:', CONFIG['split'])
if not METADATA_PATHS:
    raise RuntimeError('No model completed; inspect dependency and failure records above.')

if CONFIG['cleanup_feature_cache']:
    wm.cleanup_tabular(TABULAR_CACHE)
    print('Temporary classical feature cache cleanup requested.')
